In [1]:
import os
import pandas as pd

In [2]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, balanced_accuracy_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
## PCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from imblearn.over_sampling import SMOTE

## svm
from sklearn.svm import SVC
import pandas as pd


In [3]:
ML_CLASSIFIER = ['MLP', 'GaussianNB', "Adaboost", "KNN", "RFClassifier", "SVM_linear", "SVM_sigmoid", "SVM_RBF", "XGBoost"] # "ELM"

def classify(X_train, y_train, X_test, y_test, classifier, search_type='grid'):
    """
    Classify the data using a Random Forest Classifier
    :param X_train: Training data
    :param y_train: Training labels
    :param X_test: Testing data
    :param y_test: Testing labels
    :param search_type: Type of search to perform
    :return: Accuracy of the classifier
    """
    # Create a Random Forest Classifier
    #clf = RandomForestClassifier()

    if classifier == "MLP":
        # clf = MLPClassifier(hidden_layer_sizes=(100,100, 50), max_iter=1000) ### uncomment if the performance is not good with below hyperparameter
        clf = MLPClassifier(hidden_layer_sizes=(100,100, 50), max_iter=1000)
    elif classifier == "GaussianNB": ### {'priors': None, 'var_smoothing': 1e-05}
        clf = GaussianNB(priors=None, var_smoothing= 1e-05) ### Often, the default parameters work well for many datasets.
    elif classifier == "Adaboost":
        dtc = DecisionTreeClassifier(ccp_alpha=0.0, class_weight='balanced', criterion='entropy', max_depth=10, max_features=None, max_leaf_nodes=None, min_impurity_decrease=0.0, min_samples_leaf=2, min_samples_split=2, splitter='best')
        clf = AdaBoostClassifier(estimator=dtc, n_estimators=50, random_state=0, learning_rate=1) ###{'learning_rate': 1, 'n_estimators': 200}
        
    elif classifier == "KNN":
        clf = KNeighborsClassifier(algorithm='auto', leaf_size=10, metric='euclidean', n_jobs=-1,  n_neighbors=1, p=1, weights='uniform')  
    elif classifier == "RFClassifier": ### 
        clf = RandomForestClassifier(bootstrap = False, criterion = 'entropy', max_depth = None, max_features = 'sqrt', min_samples_leaf = 1, min_samples_split = 2, n_estimators = 100, oob_score = False, random_state = 42)
    elif classifier == "SVM_linear": ##
        clf = SVC(kernel='linear')
    elif classifier == "SVM_sigmoid": ###
        clf = SVC(kernel='sigmoid')
    elif classifier == "SVM_RBF": 
        clf = SVC(C=10, cache_size = 200, class_weight = None, gamma = 'scale', kernel = 'rbf', max_iter = -1, probability = True, shrinking = True, tol = 0.001)
    elif classifier == "XGBoost": #'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.7
        clf = XGBClassifier(n_estimators=50, learning_rate=0.1, max_depth=3,subsample=0.7)
    else:
        print("Invalid classifier")
        return None
        # Fit the model
    clf.fit(X_train, y_train)

    # Predict the test data
    y_pred = clf.predict(X_test)

    # Calculate the accuracy
    accuracy = balanced_accuracy_score(y_test, y_pred)
    
    return accuracy


In [4]:
data = pd.read_csv('BT-large-4c-dataset_results_finetune_ALL_Models_v3.csv')
data.head(15)

,Model,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF
0,resnet50,0.596488,0.782488,0.474908,0.762497,0.705696,0.761710,0.785267,0.240452,0.792667
1,resnet101,0.593891,0.727092,0.337184,0.711766,0.738522,0.766901,0.736031,0.274348,0.721348
2,densenet121,0.689558,0.726801,0.455121,0.812521,0.776924,0.754627,0.731575,0.646214,0.725370
3,densenet169,0.715670,0.712318,0.426185,0.768737,0.739897,0.738749,0.716531,0.653030,0.709357
4,vgg16,0.653996,0.749054,0.599103,0.760585,0.749019,0.742397,0.774233,0.623185,0.772703
5,vgg19,0.652916,0.749988,0.545206,0.746284,0.721936,0.736610,0.757488,0.551889,0.776554
6,alexnet,0.681697,0.733231,0.502461,0.724853,0.772667,0.739110,0.771137,0.460598,0.766272
7,resnext50_32x4d,0.597486,0.632369,0.411075,0.716078,0.732970,0.745931,0.662356,0.233913,0.699321
8,resnext101_32x8d,0.551812,0.718387,0.396856,0.715065,0.745752,0.735887,0.670952,0.227758,0.741575
9,shufflenet_v2_x1_0,0.623232,0.737218,0.562868,0.711113,0.786598,0.780967,0.753649,0.680268,0.738096


In [5]:
## taking row-wise average of all the data 
data['average'] = data.drop("Model", axis=1).mean(axis=1)


In [6]:
data.drop("Model", axis=1).mean(axis=0)

XGBoost         0.650475
MLP             0.748487
GaussianNB      0.493223
Adaboost        0.738460
KNN             0.760716
RFClassifier    0.748047
SVM_linear      0.737906
SVM_sigmoid     0.546750
SVM_RBF         0.747445
average         0.685723
dtype: float64

In [7]:
## add coumn-wise average of all the data
#data.loc['average'] = data.drop("Model", axis=1).mean(axis=0)

In [6]:
data

,Model,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF,average
0,resnet50,0.596488,0.782488,0.474908,0.762497,0.705696,0.761710,0.785267,0.240452,0.792667,0.655797
1,resnet101,0.593891,0.727092,0.337184,0.711766,0.738522,0.766901,0.736031,0.274348,0.721348,0.623009
2,densenet121,0.689558,0.726801,0.455121,0.812521,0.776924,0.754627,0.731575,0.646214,0.725370,0.702079
3,densenet169,0.715670,0.712318,0.426185,0.768737,0.739897,0.738749,0.716531,0.653030,0.709357,0.686719
4,vgg16,0.653996,0.749054,0.599103,0.760585,0.749019,0.742397,0.774233,0.623185,0.772703,0.713808
5,vgg19,0.652916,0.749988,0.545206,0.746284,0.721936,0.736610,0.757488,0.551889,0.776554,0.693208
6,alexnet,0.681697,0.733231,0.502461,0.724853,0.772667,0.739110,0.771137,0.460598,0.766272,0.683559
7,resnext50_32x4d,0.597486,0.632369,0.411075,0.716078,0.732970,0.745931,0.662356,0.233913,0.699321,0.603500
8,resnext101_32x8d,0.551812,0.718387,0.396856,0.715065,0.745752,0.735887,0.670952,0.227758,0.741575,0.611561
9,shufflenet_v2_x1_0,0.623232,0.737218,0.562868,0.711113,0.786598,0.780967,0.753649,0.680268,0.738096,0.708223


In [7]:
## sort the data by average
data = data.sort_values(by='average', ascending=False)
data

,Model,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF,average
15,vit_small_patch32_224,0.716707,0.768963,0.549337,0.776654,0.806542,0.785167,0.760776,0.599472,0.779515,0.727015
11,mnasnet0_5,0.611446,0.748422,0.671027,0.705752,0.730414,0.770967,0.745370,0.704675,0.745044,0.714791
4,vgg16,0.653996,0.749054,0.599103,0.760585,0.749019,0.742397,0.774233,0.623185,0.772703,0.713808
24,vit_base_patch32_384,0.675074,0.769932,0.496328,0.769189,0.753502,0.759932,0.777297,0.644541,0.749662,0.710606
19,vit_small_patch16_224,0.669743,0.779515,0.531658,0.742488,0.818299,0.761880,0.736022,0.584319,0.756002,0.708881
9,shufflenet_v2_x1_0,0.623232,0.737218,0.562868,0.711113,0.786598,0.780967,0.753649,0.680268,0.738096,0.708223
20,vit_base_patch16_384,0.691730,0.739932,0.492955,0.749380,0.781868,0.774841,0.741880,0.617693,0.749245,0.704392
23,vit_small_patch16_384,0.633011,0.767568,0.508307,0.766689,0.780855,0.757949,0.748096,0.626020,0.748919,0.704157
2,densenet121,0.689558,0.726801,0.455121,0.812521,0.776924,0.754627,0.731575,0.646214,0.725370,0.702079
13,vit_base_patch32_224,0.683471,0.757150,0.485991,0.743693,0.792150,0.746002,0.740405,0.630315,0.737488,0.701852


In [8]:
list(data.head(3)['Model'])

['vit_small_patch32_224', 'mnasnet0_5', 'vgg16']

### Top 3 performance model

In [9]:
top_model_combinations = [
                          ['vit_small_patch32_224', 'mnasnet0_5'],
                          ['vit_small_patch32_224', 'vgg16'],
                          ['mnasnet0_5', 'vgg16'],
                          ['vit_small_patch32_224', 'mnasnet0_5', 'vgg16']
                          ]

Ensemble vgg16 with Top 2 networks ['vit_base_patch32_384', 'vit_small_patch32_224', 'vgg16']

In [12]:



# top_model_combinations = [
#                           ['vit_base_patch32_384', 'vit_small_patch32_224'],
#                           ['vit_base_patch32_384', 'vgg16'],
#                           ['vit_small_patch32_224', 'vgg16'],
#                           ['vit_base_patch32_384', 'vit_small_patch32_224', 'vgg16']
#                           ]

In [10]:
columns = ['Model', "XGBoost", 'MLP', 'GaussianNB', "Adaboost", "KNN", "RFClassifier", "SVM_linear", "SVM_sigmoid", "SVM_RBF", ]

dataframe = pd.DataFrame(columns=columns)
# add 12 rows to the dataframe with zero values 
for model_list in top_model_combinations:
    model_name = ' + '.join(model_list)
    new_row = {'Model': model_name, "XGBoost":0, 'MLP': 0, 'GaussianNB': 0, "Adaboost": 0, "KNN": 0, "RFClassifier": 0, "SVM_linear": 0, "SVM_sigmoid": 0, "SVM_RBF": 0} 
    dataframe.loc[len(dataframe)] = new_row
    
model_versions = ['simple_version', "with_normalization_&_PCA", "with_SMOTE_only", "with_normalization_&_PCA_SMOTE"]

model_version_index = 3

main_path = 'extracted_features_BT-large-4c'
for ml_classifier in ML_CLASSIFIER:
    for model_list in top_model_combinations:
        print('Model List:', model_list)
        ensemble_X_train = []
        ensemble_X_test = []

        for model in model_list:
            sub_dir = os.path.join(main_path, model)
            # Load the data
            X_train = np.load(os.path.join(sub_dir, 'train_data_array_features.npy'))
            y_train = np.load(os.path.join(sub_dir, 'train_data_array_labels.npy'))
            X_test = np.load(os.path.join(sub_dir, 'test_data_array_features.npy'))
            y_test = np.load(os.path.join(sub_dir, 'test_data_array_labels.npy'))


            ## squeeze the dimensions 1 from the features 
            X_train = np.squeeze(X_train, axis=1)
            X_test = np.squeeze(X_test, axis=1)

            ## concatenate the features on axis 1
            ensemble_X_train.append(X_train)
            ensemble_X_test.append(X_test)
            

        X_train = np.concatenate(ensemble_X_train, axis=1)
        y_train = y_train
        X_test = np.concatenate(ensemble_X_test, axis=1)
        y_test = y_test

        # ## apply the standard scaler and PCA
        scaler = StandardScaler()
        X_train_normalized = scaler.fit_transform(X_train)
        X_test_normalized = scaler.transform(X_test)

        # # Step 2 & 3: Apply PCA
        # n_components = int(0.20* X_train.shape[1])
        # pca = PCA(n_components=n_components)
        # X_train = pca.fit_transform(X_train_normalized)
        # X_test = pca.transform(X_test_normalized)
        
        
        max_components = min(X_train.shape[0], X_train.shape[1])
        n_components = int(0.45 * max_components)  # 45% of smaller dimension

        # For automatic variance-based selection (alternative approach):
        # pca = PCA(n_components=0.95)  # Keeps 95% variance

        pca = PCA(n_components=n_components)
        X_train = pca.fit_transform(X_train_normalized)
        X_test = pca.transform(X_test_normalized)
                
        

        # ## no of example in each class
        print("no of example in each class before SMOTE:", np.bincount(y_train))


        # ## over sample the data (data augmentation)
        smote = SMOTE(sampling_strategy='minority')
        X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

        # ## no of example in each class
        print("no of example in each class after SMOTE:", np.bincount(y_resampled))


        # print("no of features in X_train:", X_train.shape)

        
        # Classify the data
        accuracy = classify(X_train, y_train, X_test, y_test, ml_classifier)
        print('Accuracy:', accuracy)
        model_name = ' + '.join(model_list)

        dataframe.loc[dataframe['Model'] == model_name, ml_classifier] = accuracy

    print(dataframe)
    dataframe.to_csv(f'BT-large-4c-dataset_results_top_three_{model_versions[model_version_index]}.csv', index=False)
    # dataframe.to_csv('BT-large-4c-dataset_results_top_three_normalized_PCA_tuned_parameters.csv', index=False)


Model List: ['vit_small_patch32_224', 'mnasnet0_5']
no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.7452790834312574
Model List: ['vit_small_patch32_224', 'vgg16']


C:\Users\user\AppData\Local\Temp\ipykernel_17624\1567469453.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.7452790834312574' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataframe.loc[dataframe['Model'] == model_name, ml_classifier] = accuracy


no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.7479494712103407
Model List: ['mnasnet0_5', 'vgg16']
no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.7260135135135135
Model List: ['vit_small_patch32_224', 'mnasnet0_5', 'vgg16']
no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.7522062279670975
                                        Model  XGBoost       MLP  GaussianNB  \
0          vit_small_patch32_224 + mnasnet0_5        0  0.745279           0   
1               vit_small_patch32_224 + vgg16        0  0.747949           0   
2                          mnasnet0_5 + vgg16        0  0.726014           0   
3  vit_small_patch32_224 + mnasnet0_5 + vgg16        0  0.752206           0   

   Adaboost  KNN  RFClassifier  SV

C:\Users\user\AppData\Local\Temp\ipykernel_17624\1567469453.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.43687580437580437' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataframe.loc[dataframe['Model'] == model_name, ml_classifier] = accuracy


no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.34204409378322415
Model List: ['mnasnet0_5', 'vgg16']
no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.4134782608695652
Model List: ['vit_small_patch32_224', 'mnasnet0_5', 'vgg16']
no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.41565217391304343
                                        Model  XGBoost       MLP  GaussianNB  \
0          vit_small_patch32_224 + mnasnet0_5        0  0.745279    0.436876   
1               vit_small_patch32_224 + vgg16        0  0.747949    0.342044   
2                          mnasnet0_5 + vgg16        0  0.726014    0.413478   
3  vit_small_patch32_224 + mnasnet0_5 + vgg16        0  0.752206    0.415652   

   Adaboost  KNN  RFClassifier  

...

no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.7164484360136534
Model List: ['mnasnet0_5', 'vgg16']
no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.689305438979352
Model List: ['vit_small_patch32_224', 'mnasnet0_5', 'vgg16']
no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.694738682782161
                                        Model  XGBoost       MLP  GaussianNB  \
0          vit_small_patch32_224 + mnasnet0_5        0  0.745279    0.436876   
1               vit_small_patch32_224 + vgg16        0  0.747949    0.342044   
2                          mnasnet0_5 + vgg16        0  0.726014    0.413478   
3  vit_small_patch32_224 + mnasnet0_5 + vgg16        0  0.752206    0.415652   

   Adaboost       KNN  RFClassifier 

C:\Users\user\AppData\Local\Temp\ipykernel_17624\1567469453.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.7450998824911869' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataframe.loc[dataframe['Model'] == model_name, ml_classifier] = accuracy


no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.7508666274970623
Model List: ['mnasnet0_5', 'vgg16']
no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.7439747356051705
Model List: ['vit_small_patch32_224', 'mnasnet0_5', 'vgg16']
no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.7589747356051704
                                        Model  XGBoost       MLP  GaussianNB  \
0          vit_small_patch32_224 + mnasnet0_5        0  0.745279    0.436876   
1               vit_small_patch32_224 + vgg16        0  0.747949    0.342044   
2                          mnasnet0_5 + vgg16        0  0.726014    0.413478   
3  vit_small_patch32_224 + mnasnet0_5 + vgg16        0  0.752206    0.415652   

   Adaboost       KNN  RFClassifie

C:\Users\user\AppData\Local\Temp\ipykernel_17624\1567469453.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.5340298528342007' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataframe.loc[dataframe['Model'] == model_name, ml_classifier] = accuracy


no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.5038921996530692
Model List: ['mnasnet0_5', 'vgg16']
no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.4864890045324828
Model List: ['vit_small_patch32_224', 'mnasnet0_5', 'vgg16']
no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.5017462648984388
                                        Model   XGBoost       MLP  GaussianNB  \
0          vit_small_patch32_224 + mnasnet0_5  0.534030  0.745279    0.436876   
1               vit_small_patch32_224 + vgg16  0.503892  0.747949    0.342044   
2                          mnasnet0_5 + vgg16  0.486489  0.726014    0.413478   
3  vit_small_patch32_224 + mnasnet0_5 + vgg16  0.501746  0.752206    0.415652   

   Adaboost       KNN  RFClas

In [11]:
n_components

3847

In [14]:
dataframe

,Model,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF
0,vit_base_patch32_384 + vit_small_patch32_224,0.736086,0.762759,0.334131,0.731610,0.771137,0.738784,0.799189,0.683773,0.754797
1,vit_base_patch32_384 + vit_small_patch16_224,0.733586,0.758041,0.333893,0.754932,0.796542,0.748919,0.779797,0.658766,0.754797
2,vit_small_patch32_224 + vit_small_patch16_224,0.740533,0.756554,0.410204,0.764571,0.812421,0.777488,0.719953,0.619690,0.765811
3,vit_base_patch32_384 + vit_small_patch32_224 +...,0.721848,0.767432,0.325231,0.747432,0.802421,0.782297,0.771554,0.665940,0.762297
